In [1]:
import os
import copy
import keras
import math
import glob
import time
import random
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm import tqdm
import tensorflow as tf
from keras.layers import *
from keras.optimizers import *
from keras import backend as K
from keras.models import Model
import matplotlib.pyplot as plt
from scipy.signal import medfilt
from collections import defaultdict
from scipy.spatial.distance import cdist
import scipy.ndimage.interpolation as inter
from tensorflow.keras.models import load_model
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.callbacks import TensorBoard
from sklearn.model_selection import KFold, StratifiedKFold
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, roc_curve
)
from sklearn.preprocessing import label_binarize

2025-04-24 22:27:12.644771: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-24 22:27:12.658530: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745551632.672606  391134 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745551632.675812  391134 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-24 22:27:12.690725: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
random.seed(1234)

class Config():
    def __init__(self):
        self.frame_l = 72 # the length of frames
        self.joint_n = 33 # the number of joints
        self.joint_d = 3 # the dimension of joints
        self.clc = 3 # the number of coarse class
        self.feat_d = 528
        self.filters = 64
        self.num_folds = 5
        self.data_dir = r"/home/nyam/Ms_Thesis/Autism/Project_v3/Stimming/DataProcessing/v2/skel_clean_aug"
        self.dataset_info = r"/home/nyam/Ms_Thesis/Autism/Project_v3/Stimming/DataProcessing/v2/dataset_info_clean_aug.txt"
C = Config()

In [3]:
#Rescale to be 64 frames
def zoom(p,target_l=64,joints_num=33,joints_dim=3):
    l = p.shape[0]
    p_new = np.empty([target_l,joints_num,joints_dim])
    for m in range(joints_num):
        for n in range(joints_dim):
            p[:,m,n] = medfilt(p[:,m,n],3)
            p_new[:,m,n] = inter.zoom(p[:,m,n],target_l/l)[:target_l]
    return p_new


def get_CG(p,C):
    M = []
    iu = np.triu_indices(C.joint_n,1,C.joint_n)
    for f in range(C.frame_l):
        #distance max
        d_m = cdist(p[f],np.concatenate([p[f],np.zeros([1,C.joint_d])]),'euclidean')
        d_m = d_m[iu]
        M.append(d_m)
    M = np.stack(M)
    return M

def normlize_range(p):
    # normolize to start point, use the center for hand case
    p[:,:,0] = p[:,:,0]-np.mean(p[:,:,0])
    p[:,:,1] = p[:,:,1]-np.mean(p[:,:,1])
    p[:,:,2] = p[:,:,2]-np.mean(p[:,:,2])
    return p

# Function to read dataset info from file
def read_dataset_info(file_path):
    dataset_info = []
    with open(file_path, "r") as info_file:
        for line in info_file:
            # Strip any leading or trailing whitespace and split the line by tabs
            info = line.strip().split("\t")
            if info[1] != "HandAction":
                dataset_info.append(info)
    return dataset_info

# Function to perform label encoding
def label_encode(dataset_info):
    label_encoder = LabelEncoder()
    labels = [info[1] for info in dataset_info]
    #print(dataset_info)
    encoded_labels = label_encoder.fit_transform(labels)

    for i, info in enumerate(dataset_info):
        dataset_info[i] = list(info)  # Convert tuple to list
        dataset_info[i][1] = encoded_labels[i]  # Modify the label
    
    return dataset_info


In [4]:

def poses_diff(x):
    H, W = x.get_shape()[1],x.get_shape()[2]
    x = tf.subtract(x[:,1:,...],x[:,:-1,...])
    x = tf.image.resize(x,size=[H,W])
    return x

def pose_motion(P,frame_l):
    P_diff_slow = Lambda(lambda x: poses_diff(x))(P)
    P_diff_slow = Reshape((frame_l,-1))(P_diff_slow)
    
    P_fast = Lambda(lambda x: x[:,::2,...])(P)
    P_diff_fast = Lambda(lambda x: poses_diff(x))(P_fast)
    P_diff_fast = Reshape((int(frame_l/2),-1))(P_diff_fast)
    return P_diff_slow,P_diff_fast

def c1D(x,filters,kernel):
    x = Conv1D(filters, kernel_size=kernel,padding='same',use_bias=False)(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)
    return x

def block(x,filters):
    x = c1D(x,filters,3)
    x = c1D(x,filters,3)
    return x

def d1D(x,filters):
    x = Dense(filters,use_bias=False)(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)
    return x

def build_FM(frame_l=90,joint_n=33,joint_d=2,feat_d=528,filters=64):
    M = Input(shape=(frame_l,feat_d))
    P = Input(shape=(frame_l,joint_n,joint_d))

    diff_slow,diff_fast = pose_motion(P,frame_l)

    x = c1D(M,filters*2,1)
    x = SpatialDropout1D(0.3)(x)
    x = c1D(x,filters,3)
    x = SpatialDropout1D(0.2)(x)
    x = c1D(x,filters,1)
    x = MaxPooling1D(2)(x)
    x = SpatialDropout1D(0.1)(x)

    x_d_slow = c1D(diff_slow,filters*2,1)
    x_d_slow = SpatialDropout1D(0.3)(x_d_slow)
    x_d_slow = c1D(x_d_slow,filters,3)
    x_d_slow = SpatialDropout1D(0.2)(x_d_slow)
    x_d_slow = c1D(x_d_slow,filters,1)
    x_d_slow = MaxPool1D(2)(x_d_slow)
    x_d_slow = SpatialDropout1D(0.1)(x_d_slow)

    x_d_fast = c1D(diff_fast,filters*2,1)
    x_d_fast = SpatialDropout1D(0.3)(x_d_fast)
    x_d_fast = c1D(x_d_fast,filters,3)
    x_d_fast = SpatialDropout1D(0.2)(x_d_fast)
    x_d_fast = c1D(x_d_fast,filters,1)
    x_d_fast = SpatialDropout1D(0.1)(x_d_fast)
    x_d_fast = MaxPool1D(1)(x_d_fast)
    x_d_fast = SpatialDropout1D(0.1)(x_d_fast)

    x = concatenate([x,x_d_slow,x_d_fast])
    
    x = block(x,filters*2) #convolution
    x = MaxPool1D(2)(x)
    x = SpatialDropout1D(0.2)(x)

    x = block(x,filters*4)
    x = MaxPool1D(2)(x)
    x = SpatialDropout1D(0.2)(x)

    x = block(x,filters*8)
    x = SpatialDropout1D(0.2)(x)
    
    
    return Model(inputs=[M,P],outputs=x)


def build_stim_Net(frame_l=64,joint_n=33,joint_d=3,feat_d=528,clc_num=3,filters=64):
    M = Input(name='M', shape=(frame_l,feat_d))
    P = Input(name='P', shape=(frame_l,joint_n,joint_d))

    FM = build_FM(frame_l,joint_n,joint_d,feat_d,filters)

    x = FM([M,P])

    x = GlobalMaxPool1D()(x)

    x = d1D(x,256)
    x = Dropout(0.5)(x)
    x = d1D(x,128)
    x = Dropout(0.5)(x)
    x = d1D(x,64)
    x = Dropout(0.5)(x)
    x = d1D(x,32)
    x = Dropout(0.5)(x)
    x = Dense(clc_num, activation='softmax')(x)
    
    model = Model(inputs=[M,P],outputs=x)
    return model

In [5]:
stim_Net = build_stim_Net(C.frame_l,C.joint_n,C.joint_d,C.feat_d,C.clc,C.filters)
lr = 1e-3
stim_Net.compile(loss="categorical_crossentropy",optimizer=keras.optimizers.Adam(lr),metrics=['accuracy'])
lrScheduler = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=10, cooldown=5, min_lr=1e-6)
stim_Net.summary()

I0000 00:00:1745551636.518677  391134 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3620 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 6GB Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6
/home/nyam/anaconda3/lib/python3.12/site-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ M (InputLayer)      │ (None, 72, 528)   │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ P (InputLayer)      │ (None, 72, 33, 3) │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional          │ (None, 9, 512)    │  1,786,624 │ M[0][0], P[0][0]  │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 512)       │          0 │ functional[0][0]  │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │    131,072 │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256)       │      1,024 │ dense[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_15      │ (None, 256)       │          0 │ batch_normalizat… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ leaky_re_lu_15[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     32,768 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ dense_1[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_16      │ (None, 128)       │          0 │ batch_normalizat… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ leaky_re_lu_16[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │      8,192 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64)        │        256 │ dense_2[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_17      │ (None, 64)        │          0 │ batch_normalizat… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 64)        │          0 │ leaky_re_lu_17[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 32)        │      2,048 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32)        │        128 │ dense_3[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_18      │ (None, 32)        │          0 │ batch_normalizat… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 32)        │          0 │ leaky_re_lu_18[0

 Total params: 1,962,723 (7.49 MB)

 Trainable params: 1,956,643 (7.46 MB)

 Non-trainable params: 6,080 (23.75 KB)

In [6]:
# Define models and their input shapes
model_dict = {
    "StimNet": {
        "model": stim_Net,
        "input_shapes": [(1, 72, 528), (1, 72, 33, 3)],
    }
}

def measure_inference_time(model, input_shapes, runs=100):
    motion_input = np.random.randn(*input_shapes[0]).astype(np.float32)
    pose_input = np.random.randn(*input_shapes[1]).astype(np.float32)
    inputs = [motion_input, pose_input]

    # Warm-up
    model.predict(inputs)

    # Timed runs
    start = time.time()
    for _ in range(runs):
        model.predict(inputs)
    end = time.time()

    return ((end - start) / runs) * 1000  # ms

def get_model_stats(model, save_path="temp_model.keras"):
    model.save(save_path, include_optimizer=False)
    size_mb = os.path.getsize(save_path) / (1024 * 1024)
    os.remove(save_path)
    param_count = model.count_params()
    return param_count, round(size_mb, 2)

# # Collect results
# results = []

# for model_name, info in model_dict.items():
#     try:
#         model = info["model"]
#         input_shapes = info["input_shapes"]

#         inf_time = measure_inference_time(model, input_shapes)
#         params, size_mb = get_model_stats(model)

#         results.append({
#             "Model": model_name,
#             "Parameters": params,
#             "Model Size (MB)": size_mb,
#             "Inference Time (ms)": round(inf_time, 2)
#         })

#     except Exception as e:
#         results.append({
#             "Model": model_name,
#             "Parameters": "Error",
#             "Model Size (MB)": "Error",
#             "Inference Time (ms)": f"Failed: {str(e)}"
#         })

# # Output results
# df = pd.DataFrame(results)
# df.to_csv("model_timing_report.csv", index=False)
# print(df)

In [7]:
# --- Visualization: Confusion Matrix ---
def plot_confusion_matrix(y_true, y_pred, labels):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    plt.figure(figsize=(6, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

In [8]:
def data_loader(dataset):
    data = {}
    data['pose'] = []
    data['label'] = []
    
    for i in tqdm(range(len(dataset))):
        label = dataset[i][1]
        skeletons = dataset[i][0]
        
        skeleton_path =os.path.join(C.data_dir, str(skeletons))
        
        try:
            p = np.load(skeleton_path).astype('float32')
            for j in range(p.shape[1]):
                p[:, j] = medfilt(p[:, j])
            
            data['pose'].append(p)
            data['label'].append(label)
        except Exception as e:
            print(f"An error occurred while processing {skeletons}: {e}")
            
    X_0 = []
    X_1 = []
    Y = []
    
    for i in tqdm(range(len(data['pose']))):
        p = np.copy(data['pose'][i]).reshape([-1,33,3])
        if(p.shape[0]>10):
            p = zoom(p,target_l=C.frame_l,joints_num=C.joint_n,joints_dim=C.joint_d)
            p = normlize_range(p)
            # print(p)
            label = np.zeros(C.clc)
            #print(int(Train['label'][i]))
            label[int(data['label'][i])] = 1
    
            M = get_CG(p,C)
    
            X_0.append(M)
            X_1.append(p)
            Y.append(label)
    
    return np.stack(X_0) , np.stack(X_1), np.stack(Y)

In [ ]:
# Define the path to the dataset info file
output_info_file = C.dataset_info  # Path to dataset info

# Load dataset info
dataset_info = read_dataset_info(output_info_file)

# Group subjects per class
classwise_subjects = defaultdict(lambda: defaultdict(list))
labels = []

for info in dataset_info:
    class_label = info[1]  # Class (e.g., "Spinning", "HeadBanging")
    filename = info[2]     # Extract filename
    parts = filename.split('_')
    subject_id = f"{parts[1]}_{parts[2]}"  # Unique subject identifier
    
    classwise_subjects[class_label][subject_id].append((filename, class_label))  # Store filename & label

subject_to_label = {}

# Iterate over classwise_subjects to populate subject labels
for class_label, subjects in classwise_subjects.items():
    for subject_id in subjects.keys():
        subject_to_label[subject_id] = class_label  # Assign class label to subject

# Collect unique subjects across all classes
all_subjects = list(subject_to_label.keys())  # Unique subjects
labels = [subject_to_label[subj] for subj in all_subjects]  # List of class labels

# Perform balanced train-test split for each class
random.seed(24)  # Ensure reproducibility
random.shuffle(all_subjects)

# Create subject-level KFold splits
kfold = StratifiedKFold(n_splits=C.num_folds, shuffle=True)  

fold_results = []
y_true_pred = []
fold_histories = []

for fold, (train_subj_idx, test_subj_idx) in enumerate(kfold.split(all_subjects, labels)):
    print(f"\n--- Fold {fold+1}/{C.num_folds} ---")

    train_subjects = set([all_subjects[i] for i in train_subj_idx])
    test_subjects = set([all_subjects[i] for i in test_subj_idx])

    # Create train and test sets ensuring mutual exclusivity
    train_list = label_encode([vid for class_subjects in classwise_subjects.values() 
                  for subj, vids in class_subjects.items() if subj in train_subjects for vid in vids])
    test_list = label_encode([vid for class_subjects in classwise_subjects.values() 
                 for subj, vids in class_subjects.items() if subj in test_subjects for vid in vids])

    x_0, x_1, y = data_loader(train_list)
    xt_0, xt_1, yt = data_loader(test_list)
    
    # Build the model
    stim_Net = build_stim_Net(C.frame_l, C.joint_n, C.joint_d, C.feat_d, C.clc, C.filters)
    lr = 1e-3
    stim_Net.compile(loss="categorical_crossentropy", optimizer=keras.optimizers.Adam(lr), metrics=['accuracy'])

    # Callbacks
    lrScheduler = ReduceLROnPlateau(monitor='loss', factor=0.1, patience=10, cooldown=5, min_lr=1e-3)
    
    checkpoint_callback = ModelCheckpoint(
        filepath=f'best_model_fold_{fold+1}.keras',  # Save per fold
        monitor='val_accuracy',  # Track validation accuracy
        save_best_only=True,  # Save only the best model
        mode='max',  # Higher accuracy is better
        verbose=1
    )
    early_stop = keras.callbacks.EarlyStopping(
        monitor='loss',
        patience=40,
        restore_best_weights=True,
        verbose=1
    )
    # Train the model with checkpointing
    history = stim_Net.fit(
        [x_0, x_1], y,
        batch_size=32,
        epochs=300,
        verbose=True,
        shuffle=True,
        callbacks=[lrScheduler, checkpoint_callback],  # Include checkpoint callback
        validation_data=([xt_0, xt_1], yt)
    )
    fold_histories.append(history.history)
    
    # Load the best model before evaluating
    stim_Net.load_weights(f'best_model_fold_{fold+1}.keras')

    # Predictions
    Y_pred = stim_Net.predict([xt_0, xt_1])
    labels = ['Arm Flapping', 'HeadBanging', 'Spinning']
    
    y_true = [labels[i] for i in np.argmax(yt, axis=1)]
    y_pred = [labels[i] for i in np.argmax(Y_pred, axis=1)]
    
    y_true_pred.append([y_true, y_pred, labels])
    
    # Compute accuracy
    accuracy = accuracy_score(y_true, y_pred)
    print("Accuracy:", accuracy)

    fold_results.append(accuracy)

# Print final results
avg_accuracy = np.mean(fold_results)
print(f"\nFinal Cross-Validation Accuracy: {avg_accuracy:.4f}")

print(fold_results)



--- Fold 1/5 ---


  0%|                                                                                           | 0/365 [00:00<?, ?it/s]/tmp/ipykernel_391134/143893866.py:8: DeprecationWarning: Please import `zoom` from the `scipy.ndimage` namespace; the `scipy.ndimage.interpolation` namespace is deprecated and will be removed in SciPy 2.0.0.
  p_new[:,m,n] = inter.zoom(p[:,m,n],target_l/l)[:target_l]
100%|██████████████████████████████████████████████████████████████████████████████████| 95/95 [00:00<00:00, 159.62it/s]
/home/nyam/anaconda3/lib/python3.12/site-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


Epoch 1/300


/home/nyam/anaconda3/lib/python3.12/site-packages/keras/src/models/functional.py:225: UserWarning: The structure of `inputs` doesn't match the expected structure: ['M', 'P']. Received: the structure of inputs=('*', '*')
  warnings.warn(
I0000 00:00:1745551648.500758  391180 service.cc:148] XLA service 0x7f851c0033e0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1745551648.500849  391180 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 3050 6GB Laptop GPU, Compute Capability 8.6
2025-04-24 22:27:28.666811: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1745551649.368828  391180 cuda_dnn.cc:529] Loaded cuDNN version 90501


 2/12 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.2734 - loss: 1.7116 

I0000 00:00:1745551659.635617  391180 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


... (9 outputs left out)

2025-04-24 22:30:57.991833: E external/local_xla/xla/service/slow_operation_alarm.cc:65] Trying algorithm eng53{k2=8,k13=1,k14=3,k18=0,k23=0} for conv (f32[11,128,1,36]{3,2,1,0}, u8[0]{0}) custom-call(f32[11,128,1,36]{3,2,1,0}, f32[128,128,1,3]{3,2,1,0}), window={size=1x3 pad=0_0x1_1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convForward", backend_config={"cudnn_conv_backend_config":{"activation_mode":"kNone","conv_result_scale":1,"leakyrelu_alpha":0,"side_input_scale":0},"force_earliest_schedule":false,"operation_queue_id":"0","wait_on_operation_queues":[]} is taking a while...
2025-04-24 22:30:58.769667: E external/local_xla/xla/service/slow_operation_alarm.cc:133] The operation took 1.777891387s
Trying algorithm eng53{k2=8,k13=1,k14=3,k18=0,k23=0} for conv (f32[11,128,1,36]{3,2,1,0}, u8[0]{0}) custom-call(f32[11,128,1,36]{3,2,1,0}, f32[128,128,1,3]{3,2,1,0}), window={size=1x3 pad=0_0x1_1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convForward", backend_c

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.3794 - loss: 1.3835  

2025-04-24 22:31:13.964207: E external/local_xla/xla/service/slow_operation_alarm.cc:65] Trying algorithm eng56{k2=0,k12=1,k13=2,k14=2,k15=0,k17=32,k18=0,k22=0,k23=0} for conv (f32[31,64,1,36]{3,2,1,0}, u8[0]{0}) custom-call(f32[31,64,1,36]{3,2,1,0}, f32[64,64,1,1]{3,2,1,0}), window={size=1x1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convForward", backend_config={"cudnn_conv_backend_config":{"activation_mode":"kNone","conv_result_scale":1,"leakyrelu_alpha":0,"side_input_scale":0},"force_earliest_schedule":false,"operation_queue_id":"0","wait_on_operation_queues":[]} is taking a while...
2025-04-24 22:31:15.762418: E external/local_xla/xla/service/slow_operation_alarm.cc:133] The operation took 2.798254781s
Trying algorithm eng56{k2=0,k12=1,k13=2,k14=2,k15=0,k17=32,k18=0,k22=0,k23=0} for conv (f32[31,64,1,36]{3,2,1,0}, u8[0]{0}) custom-call(f32[31,64,1,36]{3,2,1,0}, f32[64,64,1,1]{3,2,1,0}), window={size=1x1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convF


Epoch 1: val_accuracy improved from -inf to 0.45263, saving model to best_model_fold_2.keras
12/12 ━━━━━━━━━━━━━━━━━━━━ 42s 2s/step - accuracy: 0.3757 - loss: 1.3875 - val_accuracy: 0.4526 - val_loss: 1.0989 - learning_rate: 0.0010
Epoch 2/300
 7/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4308 - loss: 1.2106 
Epoch 2: val_accuracy did not improve from 0.45263
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.4036 - loss: 1.2532 - val_accuracy: 0.3053 - val_loss: 1.1193 - learning_rate: 0.0010
Epoch 3/300
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.3813 - loss: 1.3505
Epoch 3: val_accuracy did not improve from 0.45263
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.3806 - loss: 1.3470 - val_accuracy: 0.2526 - val_loss: 1.6112 - learning_rate: 0.0010
Epoch 4/300
 7/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4546 - loss: 1.2624 
Epoch 4: val_accuracy did not improve from 0.45263
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.4305 - loss: 1.2580 - 

In [ ]:
# Initialize
conf_matrices = []
accuracies = []
weighted_accuracies = []
precisions = []
recalls = []
weighted_f1_scores = []
auc_scores = []
ioc_scores = []

num_classes = len(y_true_pred[0][2])  # from labels
all_labels = y_true_pred[0][2]        # consistent labels

# Color palette for ROC
colors = plt.cm.Set1(np.linspace(0, 1, len(y_true_pred)))

# For ROC plotting
plt.figure(figsize=(10, 7))
for fold, data in enumerate(y_true_pred):
    y_true, y_pred, labels = data

    print(f"\n--- Fold {fold+1} Evaluation ---")

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    conf_matrices.append(cm)

    # Accuracy
    acc = accuracy_score(y_true, y_pred)
    weighted_acc = np.sum(np.diag(cm)) / np.sum(cm)
    accuracies.append(acc)
    weighted_accuracies.append(weighted_acc)

    # Precision, recall, F1
    precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, average='weighted')

    precisions.append(precision)
    recalls.append(recall)
    weighted_f1_scores.append(weighted_f1)

    # AUC (Macro)
    try:
        y_true_bin = label_binarize(y_true, classes=labels)
        y_pred_bin = label_binarize(y_pred, classes=labels)
        auc = roc_auc_score(y_true_bin, y_pred_bin, average='macro', multi_class="ovr")
        auc_scores.append(auc)

        # Macro-average ROC (manual, since you want visualization)
        fpr, tpr, _ = roc_curve(y_true_bin.ravel(), y_pred_bin.ravel())
        plt.plot(fpr, tpr, label=f'Fold {fold+1} (AUC = {auc:.2f})', color=colors[fold])
    except ValueError:
        auc_scores.append(np.nan)

    # IoC
    ioc = 1 - acc
    ioc_scores.append(ioc)

    # Fold scores
    print(f"Accuracy: {acc:.4f}")
    print(f"Weighted Accuracy: {weighted_acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"Weighted F1-score: {weighted_f1:.4f}")
    print(f"AUC: {auc:.4f}")
    print(f"IoC: {ioc:.4f}")

# ===== Plot 2: ROC Curve Summary =====
plt.plot([0, 1], [0, 1], 'k--', label='Chance')
plt.xlabel('False Positive Rate', fontsize=14)
plt.ylabel('True Positive Rate', fontsize=14)
plt.title('ROC Curve (Macro-average per Fold)', fontsize=16)
plt.legend(fontsize=12)
plt.grid(True)
plt.show()

# ===== Final Results =====
print("\n--- Final Cross-Validation Results ---")
print(f"Accuracy: Mean = {np.mean(accuracies):.4f}, Std = {np.std(accuracies):.4f}")
print(f"Weighted Accuracy: {np.mean(weighted_accuracies):.4f}")
print(f"Precision: {np.mean(precisions):.4f}")
print(f"Recall: {np.mean(recalls):.4f}")
print(f"Weighted F1-score: {np.mean(weighted_f1_scores):.4f}")
print(f"AUC: {np.nanmean(auc_scores):.4f}")
print(f"IoC: {np.mean(ioc_scores):.4f}")

# ===== Plot 1: Confusion Matrices =====
fig, axs = plt.subplots(2, 3, figsize=(18, 12))
axs = axs.flatten()

for i, cm in enumerate(conf_matrices):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=all_labels, yticklabels=all_labels,
                ax=axs[i], cbar=False, annot_kws={"size": 14})
    axs[i].set_title(f'Fold {i+1}', fontsize=18)
    axs[i].set_xlabel('Predicted', fontsize=16)
    axs[i].set_ylabel('True', fontsize=16)
    axs[i].tick_params(axis='both', labelsize=14)

if len(conf_matrices) < 6:
    fig.delaxes(axs[-1])

plt.tight_layout()
plt.show()

# Create 5 subplots (1 row, 5 columns)
fig, axs = plt.subplots(2,3, figsize=(18, 12))
axs = axs.flatten()
for i, hist in enumerate(fold_histories):
    epochs = range(1, len(hist['accuracy']) + 1)
    axs[i].plot(epochs, hist['loss'], label='Train Loss')
    axs[i].plot(epochs, hist['val_loss'], label='Val Loss')
    axs[i].set_title(f'Fold {i+1}', fontsize=14)
    axs[i].set_xlabel('Epochs')
    if i == 0:
        axs[i].set_ylabel('Loss')
    axs[i].grid(True)
    axs[i].legend()

plt.tight_layout()
plt.show()

# ===== Plot 3: Accuracy, Precision, Recall, F1 =====
folds = [f'Fold {i+1}' for i in range(len(y_true_pred))]

plt.figure(figsize=(10, 6))
plt.plot(folds, accuracies, marker='o', label='Accuracy')
plt.plot(folds, precisions, marker='s', label='Precision')
plt.plot(folds, recalls, marker='^', label='Recall')
plt.plot(folds, weighted_f1_scores, marker='d', label='F1 Score')

plt.title('Performance Metrics Across Folds', fontsize=16)
plt.xlabel('Fold', fontsize=14)
plt.ylabel('Score', fontsize=14)
plt.ylim(0, 1.05)
plt.grid(True)
plt.legend(fontsize=12)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()
plt.show()
